In [1]:
import pandas as pd 
import numpy as np
import wrds

In [3]:
db = wrds.Connection()

OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: SSL connection has been closed unexpectedly

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [3]:
# Get Compustat Annual Information

funda = db.raw_sql('''
                   SELECT c.cik,  c.sic, c.naics, f.gvkey, f.cusip, f.datadate, f.fyear,
                          f.sale,f.revt,f.cogs,f.xsga,f.dp,f.xrd,f.xad,f.ib,f.ebitda,f.ebit,f.nopi,f.spi,f.pi,f.txp,f.ni,f.txfed,f.txfo,f.txt,f.xint,
                          f.capx,f.oancf,f.dvt,f.ob,f.gdwlia,f.gdwlip,
                          f.rect,f.act,f.che,f.ppegt,f.invt,f.at,f.aco,f.intan,f.ao,f.ppent,f.gdwl,f.fatb,f.fatl,
                          f.lct,f.dlc,f.dltt,f.lt,f.dm,f.dcvt,f.cshrc,f.dcpstk,f.pstk,f.ap,f.lco,f.lo,f.drc,f.drlt,f.txdi,
                          f.ceq,f.scstkc,f.emp,f.csho,
                          f.prcc_f
                   FROM comp.company AS c 
                   JOIN comp.funda AS f
                   ON f.gvkey = c.gvkey
                   WHERE datadate >='1975-01-01'
                   AND f.indfmt='INDL'
                   AND f.datafmt='STD'
                   AND f.popsrc='D'
                   AND f.consol='C'
                   ''').reset_index(drop=True)

funda_raw = funda.copy()

# this table actually have two tables comp.company and comp.funda, and  gvkey, sic and naics are in both so have to state from which table
# drop=True discards the old index instead of inserting it back as a new column named index.             
                          

In [ ]:
# funda = funda_raw.copy().reset_index(drop=True)

In [ ]:
# Clean Compustat Annual Information

# In-SQL code 

funda['sic2'] = funda['sic'].str.slice(0,2)
funda['cnum'] = funda['cusip'].str.slice(0,6)
funda['prcc_f'] = funda['prcc_f'].abs() #  no trade occurred that day/month, so CRSP instead reports the average of the bid and ask quotes, with the value negated to flag that it's not a real trade price.
funda['mve_f'] = funda['prcc_f']*funda['csho']
funda = funda.dropna(subset=['at','prcc_f','ni'])
funda['datadate'] = pd.to_datetime(funda['datadate'])

# sort and clean up

funda = funda.sort_values(by=['gvkey','datadate']) # by sorting we enable it to keep the first one
funda = funda.drop_duplicates(subset =['gvkey','datadate']).reset_index(drop=True)

#  prep for clean-up and using time series of variables

funda['count'] = funda.groupby('gvkey').cumcount() + 1

# do some clean up, several of these variables have lots of missing values

funda['dr'] = np.where(funda['drc'].notna() & funda['drlt'].notna(), funda['drc'] + funda['drlt'], # since x + NA = NA in python
                       np.where(funda['drc'].notna() & funda['drlt'].isnull(), funda['drc'],
                                np.where(funda['drlt'].notna() & funda['drc'].isnull(), funda['drlt'], np.nan)))  

funda['dc'] = np.where(funda['dcvt'].isnull() & funda['dcpstk'].notna() & funda['pstk'].notna() & (funda['dcpstk'] > funda['pstk']), funda['dcpstk'] - funda['pstk'],
               np.where(funda['dcvt'].isnull() & funda['dcpstk'].notna() & funda['pstk'].isnull(), funda['dcpstk'],
                        funda['dcvt']))

funda['xint0'] = funda['xint'].fillna(0)
funda['xsga0'] = funda['xsga'].fillna(0) # Prof. Green agreed of this corrected version

# adding format change (not in the code)

funda['sic'] = pd.to_numeric(funda['sic'], errors='coerce')

In [5]:
# Create simple-just annual Compustat variables
# could make it computational easier with unique shift() per variable lagged but did not for readibility of variable creation

funda['bm'] = funda['ceq']/funda['mve_f']
funda['ep'] = funda['ib']/funda['mve_f']
funda['cashpr'] = ((funda['mve_f']+funda['dltt']-funda['at'])/funda['che'])
funda['dy'] = funda['dvt']/funda['mve_f']
funda['lev'] = funda['lt']/funda['mve_f']
funda['sp'] = funda['sale']/funda['mve_f']
funda['roic'] = (funda['ebit']-funda['nopi'])/(funda['ceq']+funda['lt']-funda['che'])
funda['rd_sale'] = funda['xrd']/funda['sale']
funda['rd_mve'] = funda['xrd']/funda['mve_f']
funda['agr'] = funda['at']/funda.groupby('gvkey')['at'].shift(1) - 1
funda['gma'] = (funda['revt']-funda['cogs'])/funda.groupby('gvkey')['at'].shift(1)
funda['chcsho'] = funda['csho']/funda.groupby('gvkey')['csho'].shift(1) - 1 
funda['lgr'] = funda['lt']/funda.groupby('gvkey')['lt'].shift(1)-1
funda['acc'] = np.where(funda['oancf'].notna(), 
                        (funda['ib']-funda['oancf'])/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2), 
                        ((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp']))/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2))
funda['pctacc'] = np.where(funda['oancf'].notna(),
                        np.where(funda['ib']==0, (funda['ib']-funda['oancf'])/0.01, (funda['ib']-funda['oancf'])/funda['ib'].abs()),
                        np.where(funda['ib']==0,
                                ((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp']))/0.01,
                                ((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp']))/funda['ib'].abs()))
funda['cfp'] = np.where(funda['oancf'].notna(),
                        funda['oancf']/funda['mve_f'],
                        (funda['ib']-((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp'])))/funda['mve_f'])
funda['absacc'] = funda['acc'].abs()
funda['age'] = funda['count']
funda['chinv'] = (funda['invt']-funda.groupby('gvkey')['invt'].shift(1))/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['hire'] = np.where(funda['emp'].isna() | funda.groupby('gvkey')['emp'].shift(1).isna(),
                         0,
                         (funda['emp']-funda.groupby('gvkey')['emp'].shift(1))/funda.groupby('gvkey')['emp'].shift(1))
funda['sgr'] = funda['sale']/funda.groupby('gvkey')['sale'].shift(1) - 1
funda['chpm'] = (funda['ib']/funda['sale']) - (funda.groupby('gvkey')['ib'].shift(1)/funda.groupby('gvkey')['sale'].shift(1))
funda['chato'] = (funda['sale']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)) - (funda.groupby('gvkey')['sale'].shift(1)/((funda.groupby('gvkey')['at'].shift(1)+funda.groupby('gvkey')['at'].shift(2))/2))
funda['pchsale_pchinvt'] = ((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))-((funda['invt']-funda.groupby('gvkey')['invt'].shift(1))/funda.groupby('gvkey')['invt'].shift(1))
funda['pchsale_pchrect'] = ((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))-((funda['rect']-funda.groupby('gvkey')['rect'].shift(1))/funda.groupby('gvkey')['rect'].shift(1))
funda['pchgm_pchsale'] = (((funda['sale']-funda['cogs'])-(funda.groupby('gvkey')['sale'].shift(1)-funda.groupby('gvkey')['cogs'].shift(1)))/(funda.groupby('gvkey')['sale'].shift(1)-funda.groupby('gvkey')['cogs'].shift(1)))-((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))
funda['pchsale_pchxsga'] = ((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))-((funda['xsga']-funda.groupby('gvkey')['xsga'].shift(1))/funda.groupby('gvkey')['xsga'].shift(1))
funda['depr'] = funda['dp']/funda['ppent']
funda['pchdepr'] = ((funda['dp']/funda['ppent'])-(funda.groupby('gvkey')['dp'].shift(1)/funda.groupby('gvkey')['ppent'].shift(1)))/(funda.groupby('gvkey')['dp'].shift(1)/funda.groupby('gvkey')['ppent'].shift(1))
funda['invest'] = np.where(funda['ppegt'].isnull(),
                           ((funda['ppent']-funda.groupby('gvkey')['ppent'].shift(1))+(funda['invt']-funda.groupby('gvkey')['invt'].shift(1)))/funda.groupby('gvkey')['at'].shift(1),
                           ((funda['ppegt']-funda.groupby('gvkey')['ppegt'].shift(1))+(funda['invt']-funda.groupby('gvkey')['invt'].shift(1)))/funda.groupby('gvkey')['at'].shift(1))
funda['egr'] = (funda['ceq']-funda.groupby('gvkey')['ceq'].shift(1))/funda.groupby('gvkey')['ceq'].shift(1)
funda['capx'] = np.where(funda['capx'].isnull() & (funda['count']>=2),
                         funda['ppent']-funda.groupby('gvkey')['ppent'].shift(1),
                         funda['capx'])
funda['pchcapx'] = (funda['capx']-funda.groupby('gvkey')['capx'].shift(1))/funda.groupby('gvkey')['capx'].shift(1)
funda['grcapx'] = (funda['capx']-funda.groupby('gvkey')['capx'].shift(2))/funda.groupby('gvkey')['capx'].shift(2)
funda['tang'] = (funda['che']+funda['rect']*0.715+funda['invt']*0.547+funda['ppent']*0.535)/funda['at']
funda['sin'] = np.where(funda['sic'].between(2100,2199) | funda['sic'].between(2080,2085) | funda['naics'].isin(['7132','71312','713210','71329','713290','72112','721120']),
                        1,
                        0)
funda['act'] = np.where(funda['act'].isnull(), 
                        funda['che']+funda['rect']+funda['invt'], 
                        funda['act'])
funda['lct'] = np.where(funda['lct'].isnull(), 
                        funda['ap'], 
                        funda['lct'])
funda['currat'] = funda['act']/funda['lct']
funda['pchcurrat'] = ((funda['act']/funda['lct'])-(funda.groupby('gvkey')['act'].shift(1)/funda.groupby('gvkey')['lct'].shift(1)))/(funda.groupby('gvkey')['act'].shift(1)/funda.groupby('gvkey')['lct'].shift(1))
funda['quick'] = (funda['act']-funda['invt'])/funda['lct']
funda['pchquick'] = (((funda['act']-funda['invt'])/funda['lct']) - ((funda.groupby('gvkey')['act'].shift(1)-funda.groupby('gvkey')['invt'].shift(1))/funda.groupby('gvkey')['lct'].shift(1))) / ((funda.groupby('gvkey')['act'].shift(1)-funda.groupby('gvkey')['invt'].shift(1))/funda.groupby('gvkey')['lct'].shift(1))
funda['salecash'] = funda['sale']/funda['che']
funda['salerec'] = funda['sale']/funda['rect']
funda['saleinv'] = funda['sale']/funda['invt']
funda['pchsaleinv'] = (((funda['sale']/funda['invt'])-(funda.groupby('gvkey')['sale'].shift(1)/funda.groupby('gvkey')['invt'].shift(1)))) / (funda.groupby('gvkey')['sale'].shift(1)/funda.groupby('gvkey')['invt'].shift(1))
funda['cashdebt'] = (funda['ib']+funda['dp'])/((funda['lt']+funda.groupby('gvkey')['lt'].shift(1))/2)
funda['realestate'] = np.where(funda['ppegt'].isnull(),
                               (funda['fatb']+funda['fatl'])/funda['ppent'],
                               (funda['fatb']+funda['fatl'])/funda['ppegt'])
funda['divi'] = np.where((funda['dvt'].notna() & (funda['dvt']>0)) & (funda.groupby('gvkey')['dvt'].shift(1).isnull() | (funda.groupby('gvkey')['dvt'].shift(1)==0)), 
                         1, 
                         0)
funda['divo'] = np.where((funda['dvt'].isnull() | (funda['dvt']==0)) & (funda.groupby('gvkey')['dvt'].shift(1).notna() & (funda.groupby('gvkey')['dvt'].shift(1)>0)), 
                         1, 
                         0)
funda['securedind'] = np.where(funda['dm'].notna() & (funda['dm'] != 0), 
                               1, 
                               0)
funda['secured'] = funda['dm']/funda['dltt']
funda['convind'] = np.where((funda['dc'].notna() & (funda['dc']!=0)) | (funda['cshrc'].notna() & (funda['cshrc']!=0)), 
                            1, 
                            0)
funda['grltnoa'] = ((funda['rect']+funda['invt']+funda['ppent']+funda['aco']+funda['intan']+funda['ao']-funda['ap']-funda['lco']-funda['lo'])- (funda.groupby('gvkey')['rect'].shift(1)+funda.groupby('gvkey')['invt'].shift(1)+funda.groupby('gvkey')['ppent'].shift(1)+funda.groupby('gvkey')['aco'].shift(1)+funda.groupby('gvkey')['intan'].shift(1)+funda.groupby('gvkey')['ao'].shift(1)-funda.groupby('gvkey')['ap'].shift(1)-funda.groupby('gvkey')['lco'].shift(1)-funda.groupby('gvkey')['lo'].shift(1))- ((funda['rect']-funda.groupby('gvkey')['rect'].shift(1))+(funda['invt']-funda.groupby('gvkey')['invt'].shift(1))+(funda['aco']-funda.groupby('gvkey')['aco'].shift(1))-((funda['ap']-funda.groupby('gvkey')['ap'].shift(1))+(funda['lco']-funda.groupby('gvkey')['lco'].shift(1)))-funda['dp']))/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)

rd_cond = (((funda['xrd']/funda['at']) - (funda.groupby('gvkey')['xrd'].shift(1)/funda.groupby('gvkey')['at'].shift(1))) / (funda.groupby('gvkey')['xrd'].shift(1)/funda.groupby('gvkey')['at'].shift(1))) > 0.05
funda['rd'] = np.where(rd_cond.fillna(False),
                       1,
                       0) # Prof. Green agreed of this corrected version and .fillna(False) avoids 'boolean value of NA is ambiguous' from nullable Float64 comparisons
funda['operprof'] = (funda['revt']-funda['cogs']-funda['xsga0']-funda['xint0'])/funda.groupby('gvkey')['ceq'].shift(1)
funda['ps'] = ((funda['ni']>0)
              + (funda['oancf']>0)
              + ((funda['ni']/funda['at']) > (funda.groupby('gvkey')['ni'].shift(1)/funda.groupby('gvkey')['at'].shift(1)))
              + (funda['oancf']>funda['ni'])
              + ((funda['dltt']/funda['at']) < (funda.groupby('gvkey')['dltt'].shift(1)/funda.groupby('gvkey')['at'].shift(1)))
              + ((funda['act']/funda['lct']) > (funda.groupby('gvkey')['act'].shift(1)/funda.groupby('gvkey')['lct'].shift(1)))
              + (((funda['sale']-funda['cogs'])/funda['sale']) > ((funda.groupby('gvkey')['sale'].shift(1)-funda.groupby('gvkey')['cogs'].shift(1))/funda.groupby('gvkey')['sale'].shift(1)))
              + ((funda['sale']/funda['at']) > (funda.groupby('gvkey')['sale'].shift(1)/funda.groupby('gvkey')['at'].shift(1)))
              + (funda['scstkc']==0))    

# prep for tb
funda['tr'] = np.select(
    [funda['fyear']<=1978,
     (funda['fyear']>=1979) & (funda['fyear']<=1986),
     funda['fyear']==1987,
     (funda['fyear']>=1988) & (funda['fyear']<=1992),
     funda['fyear']>=1993],
    [0.48, 0.46, 0.40, 0.34, 0.35],
    default=np.nan
)
funda['tb_1'] = np.where(funda['txfo'].isnull() | funda['txfed'].isnull(),
                          ((funda['txt']-funda['txdi'])/funda['tr'])/funda['ib'],
                          ((funda['txfo']+funda['txfed'])/funda['tr'])/funda['ib'])
tb1_override = (((funda['txfo']+funda['txfed'])>0) | (funda['txt']>funda['txdi'])) & (funda['ib']<=0)
funda['tb_1'] = np.where(tb1_override.fillna(False), 1, funda['tb_1'])

# prep for ms
funda['roa'] = funda['ni']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['cfroa'] = np.where(funda['oancf'].isnull(),
                          (funda['ib']+funda['dp'])/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2),
                          funda['oancf']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2))
funda['xrdint'] = funda['xrd']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['capxint'] = funda['capx']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['xadint'] = funda['xad']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)

# clean up for observations that do not have lagged observations to create variables (setting the first row to NA in the column where a lag was used)
req_vars = ['agr','invest','gma','chcsho','lgr','egr','chpm','chinv','hire','acc','pctacc','absacc','sgr',
            'pchsale_pchinvt','pchsale_pchrect','pchgm_pchsale','pchsale_pchxsga','pchcapx','ps','roa','cfroa', 
            'xrdint','capxint','xadint','divi','divo','grltnoa','rd','pchdepr','pchcurrat','pchquick','pchsaleinv','operprof']
funda.loc[funda['count']==1, req_vars] = np.nan 
funda.loc[funda['count']<3, ['chato','grcapx']] = np.nan # because using shift(2)

# other preparation steps for annual variables: industry adjustments
funda['chpmia'] = funda['chpm'] - funda.groupby(['sic2','fyear'])['chpm'].transform('mean')
funda['chatoia'] = funda['chato'] - funda.groupby(['sic2','fyear'])['chato'].transform('mean')
funda['indsale'] = funda.groupby(['sic2','fyear'])['sale'].transform('sum')
funda['chempia'] = funda['hire'] - funda.groupby(['sic2','fyear'])['hire'].transform('mean')
funda['bm_ia'] = funda['bm'] - funda.groupby(['sic2','fyear'])['bm'].transform('mean')
funda['pchcapx_ia'] = funda['pchcapx'] - funda.groupby(['sic2','fyear'])['pchcapx'].transform('mean')
funda['tb'] = funda['tb_1'] - funda.groupby(['sic2','fyear'])['tb_1'].transform('mean')
funda['cfp_ia'] = funda['cfp'] - funda.groupby(['sic2','fyear'])['cfp'].transform('mean')
funda['mve_ia'] = funda['mve_f'] - funda.groupby(['sic2','fyear'])['mve_f'].transform('mean')

funda['herf'] = (funda['sale']/funda['indsale'])**2
funda['herf'] = funda.groupby(['sic2','fyear'])['herf'].transform('sum')

# industry measures for ms
funda['md_roa'] = funda.groupby(['fyear','sic2'])['roa'].transform('median')
funda['md_cfroa'] = funda.groupby(['fyear','sic2'])['cfroa'].transform('median')
funda['md_xrdint'] = funda.groupby(['fyear','sic2'])['xrdint'].transform('median')
funda['md_capxint'] = funda.groupby(['fyear','sic2'])['capxint'].transform('median')
funda['md_xadint'] = funda.groupby(['fyear','sic2'])['xadint'].transform('median')

funda['m1'] = (funda['roa']>funda['md_roa']).fillna(False).astype(int)
funda['m2'] = (funda['cfroa']>funda['md_cfroa']).fillna(False).astype(int)
funda['m3'] = (funda['oancf']>funda['ni']).fillna(False).astype(int)
funda['m4'] = (funda['xrdint']>funda['md_xrdint']).fillna(False).astype(int)
funda['m5'] = (funda['capxint']>funda['md_capxint']).fillna(False).astype(int)
funda['m6'] = (funda['xadint']>funda['md_xadint']).fillna(False).astype(int) # still need to add one

# add the cpi (using the revised from 2013 annual average CPI-U - so different for year > 2013 than the SAS code)

cpi = pd.DataFrame({
    'yr':[2016,2015,2014,2013,2012,2011,2010,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000,
          1999,1998,1997,1996,1995,1994,1993,1992,1991,1990,1989,1988,1987,1986,1985,1984,
          1983,1982,1981,1980,1979,1978,1977,1976,1975,1974],
    'cpi':[240.007,237.017,236.736,232.957,229.594,224.939,218.056,214.537,215.303,207.342,201.6,195.3,
       188.9,183.96,179.88,177.1,172.2,166.6,163.00,160.5,156.9,152.4,148.2,144.5,140.3,
       136.2,130.7,124.00,118.3,113.6,109.6,107.6,103.9,99.6,96.5,90.9,82.4,72.6,65.2,
       60.6,56.9,53.8,49.3]
})
funda = funda.merge(cpi, left_on='fyear', right_on='yr', how='left')

funda['avgat'] = (funda['at']+funda.groupby('gvkey')['at'].shift(1))/2

x = funda['xsga']/funda['cpi']
decay_pow = 0.85**(funda['count']-1)
contrib = np.where(funda['count']==1, x/0.25, x/decay_pow)

W = pd.Series(contrib, index=funda.index).groupby(funda['gvkey']).cumsum()
funda['orgcap'] = (W*decay_pow)/funda['avgat'] #  * 0.85: each year, 15% of the existing organizational capital stock is assumed to depreciate away,  + xsga(t)/cpi(t): that year's new spending gets added to the stock, deflated by CPI so you're accumulating in constant (real) dollars rather than letting inflation alone make the stock look like it's growing, scaling by dividing by assets.
funda.loc[funda['count']==1, 'orgcap'] = np.nan 

In [ ]:
# Add S&P long-term issuer credit rating (comp.adsprate) for credrat / credrat_dwn

adsprate = db.raw_sql('''
                      SELECT gvkey, datadate, splticrm
                      FROM comp.adsprate
                      ''').reset_index(drop=True)

adsprate['datadate'] = pd.to_datetime(adsprate['datadate'])
adsprate['year'] = adsprate['datadate'].dt.year

funda['year'] = funda['datadate'].dt.year
funda = funda.merge(adsprate[['gvkey', 'year', 'splticrm']], on=['gvkey', 'year'], how='left')
funda = funda.sort_values(['gvkey', 'datadate'], kind='stable').drop_duplicates(subset=['gvkey', 'datadate'], keep='first').reset_index(drop=True) # kind = stable preserves original row order among ties
funda = funda.drop(columns=['year'])

credrat_map = {
    'D': 1, 'C': 2, 'CC': 3, 'CCC-': 4, 'CCC': 5, 'CCC+': 6,
    'B-': 7, 'B': 8, 'B+': 9, 'BB-': 10, 'BB': 11, 'BB+': 12,
    'BBB-': 13, 'BBB': 14, 'BBB+': 15, 'A-': 16, 'A': 17, 'A+': 18,
    'AA-': 19, 'AA': 20, 'AA+': 21, 'AAA': 22,
}
funda['credrat'] = funda['splticrm'].map(credrat_map)

credrat_lag = funda.groupby('gvkey')['credrat'].shift(1)
funda['credrat_dwn'] = (funda['credrat'] < credrat_lag).fillna(False).astype(int)
funda.loc[funda['count'] == 1, 'credrat_dwn'] = 0

In [7]:
# Create merge with CRSP - get the data

ccmlink = db.raw_sql('''
                     SELECT gvkey, lpermno, linktype, linkdt, linkenddt
                     FROM crsp.ccmxpf_linktable
                     WHERE linktype in ('LU', 'LC', 'LD', 'LF', 'LN', 'LO', 'LS', 'LX')
                     ''').reset_index(drop=True)

ccmlink_raw = ccmlink.copy()

In [ ]:
# ccmlink = ccmlink_raw.copy().reset_index(drop=True)

In [8]:
# add permno to a temporary dataset

ccmlink['linkdt'] = pd.to_datetime(ccmlink['linkdt'])
ccmlink['linkenddt'] = pd.to_datetime(ccmlink['linkenddt'])

ccmlink = ccmlink[((ccmlink['linkdt'].dt.year <= 2015) | ccmlink['linkdt'].isna()) & # missing row in linkdt means always valid
          ((ccmlink['linkenddt'].dt.year >= 1950) | ccmlink['linkenddt'].isna())] # missing row in linkenddt means still valid

temp = funda.merge(ccmlink, on='gvkey', how='inner')
temp = temp[
    ((temp['linkdt'] <= temp['datadate']) | temp['linkdt'].isna()) &
    ((temp['datadate'] <= temp['linkenddt']) | temp['linkenddt'].isna()) &
    temp['lpermno'].notna()
]
temp = temp.rename(columns={'lpermno': 'permno'})
temp = temp.drop(columns=['linktype', 'linkdt', 'linkenddt'])
temp['permno'] = temp['permno'].astype('Int64')

In [ ]:
# keep only common stocks and major exchanges

mseall = db.raw_sql('''
                    SELECT permno, date, exchcd, shrcd, siccd
                    FROM crsp.mseall
                    ''').reset_index(drop=True)

msesall_raw = mseall.copy().reset_index(drop=True)

In [ ]:
#mseall = msesall_raw.copy().reset_index(drop=True)

In [ ]:
# screen for only NYSE, AMEX, NASDAQ, and common stock, then dedupe by (permno, exchcd, date)
mseall = mseall[(mseall['exchcd'].isin([1, 2, 3])) | (mseall['shrcd'].isin([10, 11, 12]))]
mseall = mseall.sort_values(['permno', 'exchcd', 'date'], kind='stable')
mseall = mseall.drop_duplicates(subset=['permno', 'exchcd', 'date'], keep='first')

In [ ]:
# exchange start/end dates per permno-exchcd segment
mseall['exchstdt'] = mseall.groupby(['permno', 'exchcd'])['date'].transform('min')
mseall['exchedt'] = mseall.groupby(['permno', 'exchcd'])['date'].transform('max')

# collapse to one row per permno-exchcd (keeps the earliest-dated row, same as SAS's nodupkey here)
mseall = mseall.sort_values(['permno', 'exchcd', 'date'], kind='stable')
mseall = mseall.drop_duplicates(subset=['permno', 'exchcd'], keep='first')

# left join temp to mseall on permno, restricted to datadate falling within [exchstdt, exchedt]
temp = temp.merge(mseall, on='permno', how='left')
temp = temp[(temp['datadate'] >= temp['exchstdt']) & (temp['datadate'] <= temp['exchedt'])]

# keep only major-exchange common stocks (SAS comments out shrcd 12, keeping only 10/11)
temp = temp[(temp['exchcd'].isin([1, 2, 3])) & (temp['shrcd'].isin([10, 11])) & temp['permno'].notna()]
temp = temp.drop(columns=['shrcd', 'date', 'siccd', 'exchstdt', 'exchedt'])

# one row per gvkey-datadate
temp = temp.sort_values(['gvkey', 'datadate'], kind='stable').drop_duplicates(subset=['gvkey', 'datadate'], keep='first')

KeyError: 'exchcd'

In [ ]:
# Finalize first Compustat data set
# This is most of the annual compustat variables plus a couple components that still need additional information

keep_cols = [
    'gvkey', 'permno', 'exchcd', 'datadate', 'fyear', 'sic2',
    'bm', 'cfp', 'ep', 'cashpr', 'dy', 'lev', 'sp', 'roic', 'rd_sale', 'rd_mve', 'agr', 'invest', 'gma',
    'chcsho', 'lgr', 'egr', 'chpm', 'chato', 'chinv', 'hire', 'acc', 'pctacc', 'absacc', 'age',
    'sgr', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchgm_pchsale', 'pchsale_pchxsga', 'pchcapx',
    'ps', 'divi', 'divo', 'obklg', 'securedind', 'secured', 'convind', 'grltnoa',
    'rd', 'chpmia', 'chatoia', 'chempia', 'bm_ia', 'pchcapx_ia', 'tb', 'cfp_ia', 'mve_ia', 'herf',
    'orgcap', 'm1', 'm2', 'm3', 'm4', 'm5', 'm6',
    'grcapx', 'depr', 'pchdepr', 'tang',
    'sin', 'mve_f', 'currat', 'pchcurrat', 'quick', 'pchquick',
    'salecash', 'salerec', 'saleinv', 'pchsaleinv', 'cashdebt', 'realestate', 'operprof',
]
temp = temp[keep_cols]

In [ ]:
# Now align the annual Compustat variables in calendar month with the assumption that annual information is available with a lag of 6 months (if we had point-in-time we would use that)

msf = db.raw_sql('''
                 SELECT permno, date, ret, prc, shrout, vol
                 FROM crsp.msf
                 ''').reset_index(drop=True)

msf['date'] = pd.to_datetime(msf['date'])
msf['prc'] = msf['prc'].abs() 

msf_raw = msf.copy().reset_index(drop=True)

# intnx('MONTH', datadate, n) -> truncate datadate to its month, then jump n whole months
fy_month = temp['datadate'].values.astype('datetime64[M]')
window_start = (fy_month + np.timedelta64(7, 'M')).astype('datetime64[ns]') # assumes at least a 6-month lag before investors could actually know the figures (fundamentals)
window_end = (fy_month + np.timedelta64(20, 'M')).astype('datetime64[ns]') # caps how long that fiscal year's data stays "current" before it should be replaced by the next fiscal year's filing. both are  roughly a 12-13 month holding period

temp2 = temp.assign(window_start=window_start, window_end=window_end).merge(msf, on='permno', how='left')
temp2 = temp2[(temp2['date'] >= temp2['window_start']) & (temp2['date'] < temp2['window_end'])]
temp2 = temp2.drop(columns=['window_start', 'window_end'])

# The driver/backbone is msf (monthly returns) — every row in the output is anchored to a monthly observation, not a fiscal-year observation.
# The annual fundamentals (temp) get repeated/broadcast across however many consecutive months fall inside their [window_start, window_end) — so one fiscal-year's at, ceq, whatever accounting vars, get stamped onto ~12-13 monthly rows in a row. That's the "repetitive" part: the same annual number is attached to each month until the window rolls over to the next fiscal year.

In [ ]:
# Included delisted returns in the monthly returns

mseall = db.raw_sql('''
                     SELECT permno, date, dlret, dlstcd, exchcd
                     FROM crsp.mseall
                     ''').reset_index(drop=True)

# dlstcd — delisting status code: A 3-digit code from CRSP explaining why a stock disappeared from the exchange
# dlret — delisting return: The return for the final partial period, from the last trading price to whatever (if anything) shareholders received at delisting
# CRSP sometimes just doesn't report it — the stock vanishes with no final price recorded, especially for bad-outcome delistings where there was no orderly final trade.
# if you just drop those stocks or treat missing dlret as 0, you systematically overstate returns, because the stocks that silently disappear tend to be the ones that went bankrupt/got liquidated — a real loss that just isn't recorded.
# for 'bad outcome' (500, 520–584) is will be negative return

mseall['date'] = pd.to_datetime(mseall['date'])

mseall_raw = mseall.copy().reset_index(drop=True)

temp2 = temp2.merge(mseall, on=['permno', 'date'], how='left')

dlret_missing = temp2['dlret'].isna() # sometime
delisted = dlret_missing & ((temp2['dlstcd'] == 500) | temp2['dlstcd'].between(520, 584)).fillna(False)
# delisted '"was this stock delisted for a bad-outcome reason, AND does it have no recorded return that we need to fill in.'

temp2['dlret'] = temp2['dlret'].mask(delisted & temp2['exchcd'].isin([1, 2]), -0.35) # fixing NaN value for NYSE and AMEX
temp2['dlret'] = temp2['dlret'].mask(delisted & temp2['exchcd'].isin([3]), -0.55)  # fixing NaN value for NASDAQ
# .mask(cond, value) replaces entries where an external boolean condition is True (like np.where but only applying if TRUE)
# same as doing a['a']= np.where(cond, x,a['a'])

temp2['dlret'] = temp2['dlret'].mask(temp2['dlret'].notna() & (temp2['dlret'] < -1), -1.0) # Because a stock's total return can't logically go below -100% — the most you can lose owning equity is your entire investment 
temp2['dlret'] = temp2['dlret'].fillna(0)

temp2['ret'] = temp2['ret'] + temp2['dlret']
temp2['ret'] = temp2['ret'].mask(temp2['ret'].isna() & (temp2['dlret'] != 0), temp2['dlret']) # won't need dlret anymore

# proc sort by permno date descending datadate; proc sort nodupkey by permno date;
temp2 = temp2.sort_values(['permno', 'date', 'datadate'], ascending=[True, True, False]) # You want to keep only the row from the newer fiscal year (overlap in monthly accounting lag)
temp2 = temp2.drop_duplicates(subset=['permno', 'date'], keep='first').reset_index(drop=True)

In [ ]:
lag_prc = temp2.groupby('permno')['prc'].shift(1).abs()
lag_shrout = temp2.groupby('permno')['shrout'].shift(1)
temp2['mve_m'] = lag_prc * lag_shrout

temp2['mve'] = np.log(temp2['mve_m']) #  Taking the log compresses the huge range of market cap dramatically
temp2 = temp2[temp2.groupby('permno').cumcount() != 0].reset_index(drop=True)
# drop first row of all permno. will lost one row if lag other variable but harmless

In [ ]:
# now add in Compustat Quarterly and then add to the monthly aligned dataset

fundq = db.raw_sql('''
                 SELECT cusip, c.gvkey, fyearq,fqtr,datadate,rdq, sic,
                 ibq,saleq,txtq,revtq,cogsq,xsgaq,
                 atq,actq,cheq,lctq,dlcq,ppentq,
                 prccq, cshoq, ceqq,
                 seqq,pstkq,ltq,pstkrq 
                 FROM comp.company AS c 
                 JOIN comp.fundq AS f
                 ON f.gvkey = c.gvkey
                 WHERE datadate >='1975-01-01'
                 AND f.indfmt='INDL'
                 AND f.datafmt='STD'
                 AND f.popsrc='D'
                 AND f.consol='C'
                 ''').reset_index(drop=True)
# why two atq in the SAS sql pull?

fundq['datadate'] = pd.to_datetime(fundq['datadate'])
fundq['rdq'] = pd.to_datetime(fundq['rdq'])
fundq['cnum'] = fundq['cusip'].str.slice(0,6)
fundq['sic2'] = fundq['sic'].str.slice(0,2)
fundq = fundq.drop(columns='sic')
fundq['prccq'] = fundq['prccq'].abs()
fundq['mveq'] = fundq['prccq']*fundq['cshoq']
fundq = fundq.dropna(subset=['ibq'])
fundq = fundq.sort_values(by=['gvkey','datadate']).drop_duplicates(subset=['gvkey', 'datadate'],keep='first').reset_index(drop=True)

In [ ]:
# create first set of quarterly compustat variables

g = fundq.groupby('gvkey')  # used for lag

# pstk = pstkrq if present, else pstkq
fundq['pstk'] = fundq['pstkrq'].fillna(fundq['pstkq'])

# scal: seqq -> else ceqq+pstk -> else atq-ltq
fundq['scal'] = fundq['seqq']
fundq['scal'] = fundq['scal'].where(fundq['seqq'].notna(), fundq['ceqq'] + fundq['pstk'])
cond_scal2 = fundq['seqq'].isna() & (fundq['ceqq'].isna() | fundq['pstk'].isna())
fundq['scal'] = fundq['scal'].mask(cond_scal2, fundq['atq'] - fundq['ltq'])

# chtx
fundq['chtx'] = (fundq['txtq'] - g['txtq'].shift(4)) / g['atq'].shift(4)

# roaq
fundq['roaq'] = fundq['ibq'] / g['atq'].shift(1)

# roeq
fundq['roeq'] = fundq['ibq'] / g['scal'].shift(1)

# rsup
fundq['rsup'] = (fundq['saleq'] - g['saleq'].shift(4)) / fundq['mveq']

# SAS treats a MISSING numeric as -infinity in comparisons, so "if saleq<=0"
# is also TRUE whenever saleq is missing -- not just when it's <= 0.
cond_bad_sale = fundq['saleq'].isna() | (fundq['saleq'] <= 0)
den_sale = np.where(cond_bad_sale, 0.01, fundq['saleq'])

# sacc
actq_l1, cheq_l1, lctq_l1, dlcq_l1 = g['actq'].shift(1), g['cheq'].shift(1), g['lctq'].shift(1), g['dlcq'].shift(1)
num_sacc = (fundq['actq'] - actq_l1 - (fundq['cheq'] - cheq_l1)) - ((fundq['lctq'] - lctq_l1) - (fundq['dlcq'] - dlcq_l1))
fundq['sacc'] = num_sacc / den_sale

# stdacc
fundq['stdacc'] = g['sacc'].transform(lambda s: s.rolling(16, min_periods=2).std())

# sgrvol
fundq['sgrvol'] = g['rsup'].transform(lambda s: s.rolling(15, min_periods=2).std())

# roavol
fundq['roavol'] = g['roaq'].transform(lambda s: s.rolling(16, min_periods=2).std())

# scf (reuses sacc, which is already adjusted for the saleq<=0/missing case)
fundq['scf'] = fundq['ibq'] / den_sale - fundq['sacc']

# stdcf
fundq['stdcf'] = g['scf'].transform(lambda s: s.rolling(16, min_periods=2).std())

# cash
fundq['cash'] = fundq['cheq'] / fundq['atq']

# cinvest: SAS branches on the CURRENT row's saleq only -- if it's bad/missing,
# EVERY denominator in the formula (including the lagged-sale ones) becomes 0.01.
ppent_l1, ppent_l2, ppent_l3, ppent_l4 = g['ppentq'].shift(1), g['ppentq'].shift(2), g['ppentq'].shift(3), g['ppentq'].shift(4)
sale_l1, sale_l2, sale_l3 = g['saleq'].shift(1), g['saleq'].shift(2), g['saleq'].shift(3)

den0 = den_sale
den1 = np.where(cond_bad_sale, 0.01, sale_l1)
den2 = np.where(cond_bad_sale, 0.01, sale_l2)
den3 = np.where(cond_bad_sale, 0.01, sale_l3)

term0 = (fundq['ppentq'] - ppent_l1) / den0
comp1 = (ppent_l1 - ppent_l2) / den1
comp2 = (ppent_l2 - ppent_l3) / den2
comp3 = (ppent_l3 - ppent_l4) / den3

# SAS mean() skips missing args; pd.concat(...).mean(axis=1) does the same by default
mean_term = pd.concat([comp1, comp2, comp3], axis=1).mean(axis=1)
fundq['cinvest'] = term0 - mean_term

# che (income change) -- used later for sue/nincr
fundq['che'] = fundq['ibq'] - g['ibq'].shift(4)

# nincr: length of the consecutive-earnings-increase streak, capped at 8 quarters
ibq_lags = {k: g['ibq'].shift(k) for k in range(9)}
c = {k: (ibq_lags[k-1] > ibq_lags[k]).fillna(False).astype(int) for k in range(1, 9)}

fundq['nincr'] = (
    c[1]
    + c[1]*c[2]
    + c[1]*c[2]*c[3]
    + c[1]*c[2]*c[3]*c[4]
    + c[1]*c[2]*c[3]*c[4]*c[5]
    + c[1]*c[2]*c[3]*c[4]*c[5]*c[6]
    + c[1]*c[2]*c[3]*c[4]*c[5]*c[6]*c[7]
    + c[1]*c[2]*c[3]*c[4]*c[5]*c[6]*c[7]*c[8]
)

fundq['count'] = fundq.groupby('gvkey').cumcount() + 1

# first.gvkey (== count==1) -> null roaq/roeq
fundq.loc[fundq['count'] == 1, ['roaq', 'roeq']] = np.nan # because of lag

# count<5 -> null chtx/che/cinvest (these use up to lag(4))
fundq.loc[fundq['count'] < 5, ['chtx', 'che', 'cinvest']] = np.nan

# count<17 -> null the rolling-std measures (these need up to a 16-quarter window)
fundq.loc[fundq['count'] < 17, ['stdacc', 'stdcf', 'sgrvol', 'roavol']] = np.nan

In [ ]:
# finally finish Mohanram score components

fundq['md_roavol'] = fundq.groupby(['fyearq', 'fqtr', 'sic2'])['roavol'].transform('median')
fundq['md_sgrvol'] = fundq.groupby(['fyearq', 'fqtr', 'sic2'])['sgrvol'].transform('median')

# SAS treats missing numeric as -infinity in comparisons: "roavol<md_roavol" is TRUE
# whenever roavol is missing and md_roavol isn't -- not just when roavol is a smaller real number.
m7_cond = (fundq['roavol'] < fundq['md_roavol']) | (fundq['roavol'].isna() & fundq['md_roavol'].notna())
fundq['m7'] = m7_cond.astype(int)

m8_cond = (fundq['sgrvol'] < fundq['md_sgrvol']) | (fundq['sgrvol'].isna() & fundq['md_sgrvol'].notna())
fundq['m8'] = m8_cond.astype(int)

fundq = fundq.dropna(subset=['rdq'])

# adding permno via ccmlink

fundq = fundq.merge(ccmlink, on='gvkey', how='inner')
fundq = fundq[
    ((fundq['linkdt'] <= fundq['datadate']) | fundq['linkdt'].isna()) &
    ((fundq['datadate'] <= fundq['linkenddt']) | fundq['linkenddt'].isna()) &
    fundq['lpermno'].notna()
]
fundq = fundq.rename(columns={'lpermno': 'permno'}).drop(columns=['linktype', 'linkdt', 'linkenddt'])
fundq['permno'] = fundq['permno'].astype('Int64')

NameError: name 'fundq' is not defined